### This script is used to find Open Reading Frames (ORFs) and translate DNA sequences to protein sequences.

## Background:
1. ORFs are Open Reading Frames, which are continuous stretchs of codons that begins with a start codon (usually ATG) and ends at a stop codon (usually TAA, TAG or TGA).
2. There are 3 possible starting points to read the sequence, and the sequence can be read forwardly or reversely.
3. Longer ORFs are less likely to be occur by chance, and more likely to be true ORFs.
4. The average length of genes in Bacteria is ~1000 nt.
5. Nested ORFs are less likely to be true ORFs and should be ignored.
6. Overlapping genes are more likely to be false positives.
7. There are possibility of sequencing errors, so any sense codon should be considered as a start codon.

## To ensure we obtain a usable protein sequence, it’s often helpful to:
1. Translate all three possible reading frames for each DNA strand.
2. Identify the frame that produces the longest AA sequence without internal stop codons.
3. Trim the translation, keeping only the longest single unbroken AA sequence.

In [7]:
# Import required packages
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import gc_fraction

# Load slected representative sequences from the FASTA file
sequences = list(
    SeqIO.parse(
        "../Groups identification and representatives selection/"
        "representative_sequences.fasta",
        "fasta",
    )
)

# 1. DNA Sequence Statistics

In [8]:
# Calculate the length of each sequence
lengths = [len(i.seq) for i in sequences]
print(lengths)

# Calculate GC content
for seq in sequences:
    gc_percent = gc_fraction(seq.seq) * 100
    print(f"{seq.id}: GC content = {gc_percent:.2f}%")

# Calculate the number of start codon (ATG) and stop codons (TAA, TAG or TGA).
codons = ("ATG", "TAA", "TAG", "TGA")
for seq in sequences:
    sequence = str(seq.seq).upper()
    codon_counts = {
        codon: sum(sequence[i:i + 3] == codon for i in range(len(sequence) - 2))
        for codon in codons
    }

    start_count = codon_counts["ATG"]
    stop_count = sum(codon_counts[codon] for codon in codons[1:])
    print(
        f"{seq.id}: ATG occurs {start_count} times; "
        f"stop codons occur {stop_count} times "
        f"(TAA: {codon_counts['TAA']}, "
        f"TAG: {codon_counts['TAG']}, "
        f"TGA: {codon_counts['TGA']})"
    )

# Calculate number of gaps and ambiguous bases
ambiguous_bases = set("NRYSWKMBDHV")  # IUPAC ambiguity codes

# We need to loop through our sequences list. Consequently, it will be useful to transform the list into an enumerator object. This creates index-element pairs
seq_iter = enumerate(sequences, start=1)

for n, seq in seq_iter:

    # Count ambiguous bases
    num_ambiguous = 0
    for base in seq:
        if base in ambiguous_bases:
            num_ambiguous += 1

    print(f"\n=== {n} ===")
    print(sequences[n-1].id)
    print("Number of ambiguous bases:", num_ambiguous)

[3552, 3564, 3507, 3489, 3615]
outlet_01_seq068: GC content = 38.94%
outlet_02_seq053: GC content = 61.64%
outlet_05_seq033: GC content = 66.89%
outlet_05_seq074: GC content = 64.63%
outlet_08_seq092: GC content = 46.06%
outlet_01_seq068: ATG occurs 105 times; stop codons occur 231 times (TAA: 78, TAG: 51, TGA: 102)
outlet_02_seq053: ATG occurs 47 times; stop codons occur 74 times (TAA: 15, TAG: 2, TGA: 57)
outlet_05_seq033: ATG occurs 36 times; stop codons occur 46 times (TAA: 3, TAG: 3, TGA: 40)
outlet_05_seq074: ATG occurs 52 times; stop codons occur 47 times (TAA: 8, TAG: 1, TGA: 38)
outlet_08_seq092: ATG occurs 91 times; stop codons occur 203 times (TAA: 62, TAG: 24, TGA: 117)

=== 1 ===
outlet_01_seq068
Number of ambiguous bases: 0

=== 2 ===
outlet_02_seq053
Number of ambiguous bases: 0

=== 3 ===
outlet_05_seq033
Number of ambiguous bases: 0

=== 4 ===
outlet_05_seq074
Number of ambiguous bases: 0

=== 5 ===
outlet_08_seq092
Number of ambiguous bases: 0


# 2. Identifing plausible ORFs
settings:
1. Minimal ORF length is 75 nt;
2. Genetic code is 11 - bacteria;
3. Accept any sense codon as a start codon;
4. Ignore nested ORFs.

In [ ]:
min_ORF_length = 75  # nucleotides, including the stop codon
gene_code = 11  # bacterial genetic code
stop_codon = {"TAA", "TAG", "TGA"}


def identify_orfs(record):
    """Return maximal, complete ORFs in all six reading frames."""
    original_sequence = str(record.seq).upper()
    sequence_length = len(original_sequence)
    strands = (
        ("+", original_sequence),
        ("-", str(Seq(original_sequence).reverse_complement()).upper()),
    )

    orfs = []

    for strand, sequence in strands:
        for frame in range(3):
            start = None

            for position in range(frame, len(sequence) - 2, 3):
                codon = sequence[position:position + 3]
                is_sense_codon = (
                    set(codon) <= set("ACGT") and codon not in stop_codon
                )

                # The first sense codon after a stop is the maximal start.
                if start is None and is_sense_codon:
                    start = position

                if codon in stop_codon:
                    if start is not None:
                        end = position + 3
                        length = end - start

                        if length >= min_ORF_length:
                            dna_orf = sequence[start:end]

                            if strand == "+":
                                original_start = start
                                original_end = end
                                frame_name = f"+{frame + 1}"
                            else:
                                original_start = sequence_length - end
                                original_end = sequence_length - start
                                frame_name = f"-{frame + 1}"

                            orfs.append(
                                {
                                    "sequence_id": record.id,
                                    "strand": strand,
                                    "frame": frame_name,
                                    "start": original_start + 1,
                                    "end": original_end,
                                    "length": length,
                                    "dna": dna_orf,
                                    "protein": str(
                                        Seq(dna_orf).translate(
                                            table=gene_code,
                                            to_stop=False,
                                        )
                                    ),
                                }
                            )

                    # Reset after the stop: nested starts are not reported.
                    start = None

    return orfs


all_orfs = []

for record in sequences:
    record_orfs = identify_orfs(record)
    all_orfs.extend(record_orfs)

    print(f"\n{record.id}: {len(record_orfs)} ORF(s) >= {min_ORF_length} nt")
    for orf in sorted(record_orfs, key=lambda item: item["length"], reverse=True):
        print(
            f"  {orf['frame']}  {orf['start']}..{orf['end']}  "
            f"length={orf['length']} nt  protein_length={len(orf['protein'])} aa"
        )

# print the number of complete ORFs across all sequences
print(f"\nTotal complete ORFs across all sequences: {len(all_orfs)}")


outlet_01_seq068: 67 ORF(s) >= 75 nt
  +1  1..3552  length=3552 nt  protein_length=1184 aa
  -2  3255..3548  length=294 nt  protein_length=98 aa
  -2  471..707  length=237 nt  protein_length=79 aa
  -3  2939..3175  length=237 nt  protein_length=79 aa
  -3  527..751  length=225 nt  protein_length=75 aa
  -3  1808..2014  length=207 nt  protein_length=69 aa
  -2  3051..3254  length=204 nt  protein_length=68 aa
  -2  1932..2123  length=192 nt  protein_length=64 aa
  -1  1897..2079  length=183 nt  protein_length=61 aa
  +2  3116..3289  length=174 nt  protein_length=58 aa
  -3  1406..1579  length=174 nt  protein_length=58 aa
  -2  861..1031  length=171 nt  protein_length=57 aa
  +2  2930..3097  length=168 nt  protein_length=56 aa
  -3  3176..3340  length=165 nt  protein_length=55 aa
  -3  1163..1321  length=159 nt  protein_length=53 aa
  -2  1155..1307  length=153 nt  protein_length=51 aa
  -2  2487..2636  length=150 nt  protein_length=50 aa
  +2  2726..2872  length=147 nt  protein_length=4

# 3. Selcting the Most Possible ORF of Each Representative Sequence
Longer ORFs are more likely to be true ORFs, so our goal in this part is selecting the longest ORF for each representative sequence from the possible ORFs acquired from part 2.

In [11]:
selected_orfs = []

for record in sequences:
    record_orfs = [
        orf for orf in all_orfs
        if orf["sequence_id"] == record.id
    ]

    if record_orfs:
        longest_orf = max(record_orfs, key=lambda orf: orf["length"])
        selected_orfs.append(longest_orf)

        print(
            f"{record.id}: longest ORF is "
            f"{longest_orf['length']} nt, "
            f"{longest_orf['start']}..{longest_orf['end']}, "
            f"strand {longest_orf['strand']}, "
            f"frame {longest_orf['frame']}"
        )
    else:
        print(f"{record.id}: no ORF >= {min_ORF_length} nt")

# Inspect the selected ORFs
for orf in selected_orfs:

    dna = orf["dna"]
    protein = orf["protein"]

    print(f"\nSequence: {orf['sequence_id']}")
    print(f"First codon: {dna[:3]}")
    print(f"Last codon: {dna[-3:]}")
    print(f"DNA length: {len(dna)} nt")
    print(f"Protein length: {len(protein)} aa")
    print(f"Internal stop codons: {protein[:-1].count('*')}")
    print(f"Ends with stop: {protein.endswith('*')}")


outlet_01_seq068: longest ORF is 3552 nt, 1..3552, strand +, frame +1
outlet_02_seq053: longest ORF is 3564 nt, 1..3564, strand +, frame +1
outlet_05_seq033: longest ORF is 3507 nt, 1..3507, strand +, frame +1
outlet_05_seq074: longest ORF is 3489 nt, 1..3489, strand +, frame +1
outlet_08_seq092: longest ORF is 3615 nt, 1..3615, strand +, frame +1

Sequence: outlet_01_seq068
First codon: TTG
Last codon: TAA
DNA length: 3552 nt
Protein length: 1184 aa
Internal stop codons: 0
Ends with stop: True

Sequence: outlet_02_seq053
First codon: TTG
Last codon: TGA
DNA length: 3564 nt
Protein length: 1188 aa
Internal stop codons: 0
Ends with stop: True

Sequence: outlet_05_seq033
First codon: GTG
Last codon: TGA
DNA length: 3507 nt
Protein length: 1169 aa
Internal stop codons: 0
Ends with stop: True

Sequence: outlet_05_seq074
First codon: TTG
Last codon: TGA
DNA length: 3489 nt
Protein length: 1163 aa
Internal stop codons: 0
Ends with stop: True

Sequence: outlet_08_seq092
First codon: GTG
Last 

The length of the longest ORF in each sequence is equal to the length of the sequence. All of them begin with recognised bacterial initiation codons (TTG or GTG), terminate with TAA or TGA, and contain no internal stop codons, which support them to be plausible true ORFs. 

# 4. Translating DNA to Protein
Since we have identified that all of the selected most appropriate ORFs are ended with a stop codon, the transltion from DNA to protein should use the reading frame which ends with a stop codon, rather than translating with all of the 3 reading frames.

In [12]:
aa_sequences = []   # List to store output amino-acid SeqRecord objects

for orf in selected_orfs:
    dna_orf = orf["dna"]

    # Translate with stop codons retained as "*".
    protein = str(Seq(dna_orf).translate(table=gene_code, to_stop=False))

    # Complete ORFs should end in a stop codon.
    if dna_orf[-3:].upper() in stop_codon:
        if not protein.endswith("*"):
            protein += "*"

    aa_record = SeqRecord(
        Seq(protein),
        id=orf["sequence_id"],
        name=orf["sequence_id"],
        description=(
            f"selected longest ORF; strand={orf['strand']}; frame={orf['frame']}; "
            f"coordinates={orf['start']}..{orf['end']}"
        ),
        annotations={
            "genetic_code": gene_code,
            "type": "selected longest ORF",
            "strand": orf["strand"],
            "frame": orf["frame"],
            "dna_start": orf["start"],
            "dna_end": orf["end"],
            "ends_with_stop": protein.endswith("*"),
        },
    )

    aa_sequences.append(aa_record)

# Write protein sequences with terminal stop symbols preserved.
SeqIO.write(aa_sequences, "translated.fas", "fasta")

for aa_record in aa_sequences:
    print(
        f"{aa_record.id}: {len(aa_record.seq)} aa; "
        f"ends with stop = {str(aa_record.seq).endswith('*')}"
    )

outlet_01_seq068: 1184 aa; ends with stop = True
outlet_02_seq053: 1188 aa; ends with stop = True
outlet_05_seq033: 1169 aa; ends with stop = True
outlet_05_seq074: 1163 aa; ends with stop = True
outlet_08_seq092: 1205 aa; ends with stop = True
